<a href="https://colab.research.google.com/github/charre2021/REWOO_example/blob/main/REWOO_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langgraph langchain_openai tavily tavily-python

In [ ]:
import re
import os
from google.colab import userdata
from typing import List
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, StateGraph, START

In [ ]:
tavily_client = TavilyClient(userdata.get("tavily"))
model = ChatOpenAI(model = "gpt-4o", api_key = userdata.get("openai"))

prompt = """
For the following task, create a series of plans
that can solve the problem step-by-step. For each plan, specify
which external tool and its corresponding input should be used
to gather evidence. You can store the evidence in a variable #E
(e.g., #E1, #E2, #E3, etc.) that can be referenced by subsequent
tools. Note that all the variables are independent, so make sure
to include all necessary information in each tool input.

Tools can be one of the following:

Google[input]: A search engine worker that retrieves results
from Google. Use this when you need concise answers or
information about a specific topic. The input should be a search
query.

LLM[input]: A pretrained Large Language Model (like me). Use
this when you need to leverage general world knowledge, common
sense, or perform complex reasoning. The input can be any instruction or question.

Calculator[input]: A tool that can perform mathematical
calculations. Use this when you need to perform arithmetic
operations. The input should be a valid mathematical expression.

WolframAlpha[input]: A computational knowledge engine. Use this
when you need to solve equations, perform symbolic calculations,
or get data-driven answers. The input should be a query in
Wolfram Language or natural language related to a math or
science problem.

For example,
Task: Alice, Bob, and Carol earned a total of $540 from their
part-time jobs last week. Alice earned y dollars. Bob earned $20
more than three times what Alice earned, and Carol earned $15
more than Bob. How much money did Carol earn?

Plan: Given Alice earned y dollars, translate the problem into
algebraic expressions and solve with Wolfram Alpha.

#E1 = WolframAlpha[Solve y + (3y + 20) + ((3y + 20) + 15) = 540]

Plan: Find out the amount of money Alice earned.

#E2 = LLM[What is y, given #E1]

Plan: Calculate the amount of money Carol earned.

#E3 = Calculator[((3 * #E2) + 20) + 15]

Begin!

Describe your plans with rich details. Each Plan should be
followed by only one #E.

Task: {task}"""

solve_prompt = """
Solve the following task or problem. To solve
the problem, we have made step-by-step Plan and retrieved corresponding
Evidence to each Plan. Use them with caution since long evidence might
contain irrelevant information.

{plan}

Now solve the question or task according to provided Evidence above.
Respond with the answer directly with no extra words.

Task: {task}

Response:"""

In [ ]:
class ReWOO(TypedDict):
    task: str
    plan_string: str
    steps: List
    results: dict
    result: str

prompt_template = ChatPromptTemplate.from_messages(
    [("user", prompt)]
)

planner = prompt_template | model

def get_plan(state: ReWOO) -> dict:
    task = state["task"]
    result = planner.invoke({"task" : task})
    plan_pattern = r"Plan\s*\d*:\s*(.+)"
    plans = re.findall(plan_pattern, result.content)
    step_name_pattern = r"(#E\d+)\s+="
    step_names = re.findall(step_name_pattern, result.content)
    tool_name_pattern = r"#E\d+\s+=\s+(.*)\["
    tool_names = re.findall(tool_name_pattern, result.content)
    formula_pattern = r"\[(.*)\]"
    formulae = re.findall(formula_pattern, result.content)
    all_steps = list(zip(plans, step_names, tool_names, formulae))
    return {"steps" : all_steps, "plan_string": result.content}

def _get_current_task(state: ReWOO):
    if "results" not in state or state["results"] is None:
        return 1
    elif len(state["results"]) == len(state["steps"]):
        return None
    else:
        return len(state["results"]) + 1

def tool_execution(state: ReWOO):
    _step = _get_current_task(state)
    _, step_name, tool, tool_input = state["steps"][_step - 1]
    _results = (state["results"] or {}) if "results" in state else {}
    for k, v in _results.items():
        tool_input = tool_input.replace(k, v)
    if tool == "Google":
        result = tavily_client.search(query = tool_input)
    elif tool == "LLM":
        result = model.invoke(tool_input)
    else:
        raise ValueError
    _results[step_name] = str(result)
    return {"results" : _results}

def solve(state: ReWOO):
    plan = ""
    for _plan, step_name, tool, tool_input in state["steps"]:
        _results = (
            (state["results"] or {}) if "results" in state else {}
        )
        for k, v in _results.items():
            tool_input = tool_input.replace(k, v)
            step_name = step_name.replace(k, v)
        plan += (
            f"Plan: {_plan}\n"
            f"{step_name} = {tool}[{tool_input}]\n"
        )

    prompt = solve_prompt.format(plan = plan, task = state["task"])
    result = model.invoke(prompt)
    return {"result": result.content}

def _route(state):
    _step = _get_current_task(state)
    return "solve" if _step is None else "tool"

In [ ]:
graph = StateGraph(ReWOO)
graph.add_node("plan", get_plan)
graph.add_node("tool", tool_execution)
graph.add_node("solve", solve)
graph.add_edge("plan", "tool")
graph.add_conditional_edges("tool", _route)
graph.add_edge(START, "plan")

app = graph.compile()

In [ ]:
task = """
How many tennis balls would fit in standard elevator?
"""

for s in app.stream({"task" : task}):
    print(s)
    print("---")